# LIGHTNING TEST

### We had flash-attention compatibility issues with the new runtime so mask it out.  
uninstalling it didn't work because it was baked into the env.

In [0]:
import os
# Prevent TensorFlow spam
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Disable optional backends that trigger builds
os.environ["DISABLE_DEEPSPEED"] = "1"
os.environ["DISABLE_TRITON"] = "1"
os.environ["DISABLE_FLASH_ATTENTION"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

Have to run above before importing anything like transformers or lightning. So re

In [0]:
import torch, sys, subprocess, os
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda (torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())

# nvcc (nice-to-have; may not exist on managed images)
try:
    print("nvcc:", subprocess.check_output(["nvcc", "--version"]).decode().strip().splitlines()[-1])
except Exception as e:
    print("nvcc: n/a", e)

In [0]:
import os
import torch
import torch.distributed as dist
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl

class TinyNet(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x): return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, prog_bar=True, rank_zero_only=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


def train_lightning():
    if not dist.is_initialized():
        dist.init_process_group("nccl")

    world_size = dist.get_world_size()
    rank = dist.get_rank()

    # each Databricks process sees 1 GPU → no set_device()
    print(f"🌍 WORLD_SIZE={world_size}, RANK={rank}, DEVICES_VISIBLE={torch.cuda.device_count()}")

    # synthetic data
    X = torch.randn(4096, 32)
    y = torch.randint(0, 10, (4096,))
    loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)

    model = TinyNet()

    log_dir = "/local_disk0/lightning_logs"

    trainer = pl.Trainer(
        accelerator="gpu",
        devices=1,             # 1 GPU per process
        num_nodes=world_size,  # effectively 16 processes = 16 "nodes" from Lightning's view
        strategy="ddp",
        default_root_dir=log_dir,
        max_epochs=2,
        log_every_n_steps=10,
        enable_progress_bar=(rank == 0),
    )

    trainer.fit(model, loader)

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

# Adjust for your cluster size (e.g., 4 nodes × 4 GPUs)
distributor = TorchDistributor(
    num_processes=16,       # total processes = num_nodes × gpus_per_node
    local_mode=False,       # must be False for multi-node
    use_gpu=True,
)

distributor.run(train_lightning)